<a href="https://colab.research.google.com/github/varshasisodiya16-arch/first-repository/blob/main/Lab_6_4_Databases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3                    # built-in Python module to work with SQLite
import pandas as pd               # tables (DataFrame)
import sqlalchemy as sqla         # toolkit that connects pandas to many databases
import os                         # to delete the file at the end

print("SQLite version   :", sqlite3.sqlite_version)
print("SQLAlchemy version:", sqla.__version__)

SQLite version   : 3.45.1
SQLAlchemy version: 2.1.6


### A1. Connect to the database

In [4]:

if os.path.exists("mydata.sqlite"):
    os.remove("mydata.sqlite")
con = sqlite3.connect("mydata.sqlite")

print("Database file created:", os.path.exists("mydata.sqlite"))

Database file created: True


In [5]:

query = """
CREATE TABLE person
(id INTEGER PRIMARY KEY,
name VARCHAR(50),
age INTEGER
);"""

con.execute(query)     # run the command
con.commit()           # save the changes

print("Table 'test' created!")

Table 'test' created!


In [7]:
# Our data: a list of tuples (each tuple = one row)
data = [(10,     "varsha",    1),
        (11, "kaveri",  3),
        (12,  "kanak",   5)]
stmt = "INSERT INTO person VALUES(?, ?, ?)"

con.executemany(stmt, data)    # insert all rows
con.commit()                   # save

print("3 rows inserted!")

3 rows inserted!


In [8]:
cursor = con.execute("SELECT * FROM person")   # ask for everything in the table
rows = cursor.fetchall()                     # collect all rows

rows

[(10, 'varsha', 1), (11, 'kaveri', 3), (12, 'kanak', 5)]

In [9]:
# Each item in description is a tuple; the first value is the column name
cursor.description

(('id', None, None, None, None, None, None),
 ('name', None, None, None, None, None, None),
 ('age', None, None, None, None, None, None))

In [10]:
# Take only the first item (name) from each tuple
col_names = [x[0] for x in cursor.description]
print(col_names)

['id', 'name', 'age']


In [11]:
# Now build the DataFrame manually
df_manual = pd.DataFrame(rows, columns=col_names)

df_manual

,id,name,age
0,10,varsha,1
1,11,kaveri,3
2,12,kanak,5


In [13]:
db = sqla.create_engine("sqlite:///mydata.sqlite")
pd.read_sql("SELECT * FROM person", db)

,id,name,age
0,10,varsha,1
1,11,kaveri,3
2,12,kanak,5


In [14]:
# Check the data types of the result
df = pd.read_sql("SELECT * FROM person", db)
df.dtypes

,0
id,int64
name,object
age,int64


In [18]:
data3 = [(1," chhavi",23),
             (2," vishaka",45),
             (3,"chaya",33)
             ]

con.executemany("INSERT OR IGNORE INTO person VALUES(?, ?, ?)", data3)
con.commit()

pd.read_sql("SELECT * FROM person", db)

,id,name,age
0,1,chhavi,23
1,2,vishaka,45
2,3,chaya,33
3,10,varsha,1
4,11,kaveri,3
5,12,kanak,5


In [20]:
pd.read_sql("SELECT name, age FROM person", db)

,name,age
0,chhavi,23
1,vishaka,45
2,chaya,33
3,varsha,1
4,kaveri,3
5,kanak,5


In [21]:
pd.read_sql("SELECT * FROM person WHERE age > 20", db)

,id,name,age
0,1,chhavi,23
1,2,vishaka,45
2,3,chaya,33


### E3. Sort with `ORDER BY`
Highest rainfall first (`DESC` = descending):

In [23]:
pd.read_sql("SELECT name,age  FROM person  ORDER BY age DESC", db)

,name,age
0,vishaka,45
1,chaya,33
2,chhavi,23
3,kanak,5
4,kaveri,3
5,varsha,1


### E4. Group and calculate with `GROUP BY`
Average rainfall for each state (same idea as pandas `groupby`):

In [24]:
pd.read_sql("""
SELECT name, AVG(age) AS avg_age, COUNT(*) AS num_name
FROM person
GROUP BY age
ORDER BY age DESC
""", db)

,name,avg_age,num_name
0,vishaka,45.0,1
1,chaya,33.0,1
2,chhavi,23.0,1
3,kanak,5.0,1
4,kaveri,3.0,1
5,varsha,1.0,1



We can also do the reverse: take a pandas DataFrame and store it as a **new table** using `df.to_sql("table_name", engine)`.

In [25]:
# A small student DataFrame
students = pd.DataFrame({
    "roll_no": [101, 102, 103, 104],
    "name":    ["Aman", "Bhavna", "Chirag", "Divya"],
    "marks":   [78.5, 85.0, 62.0, 91.5],
})

# Save it as a table named 'students'
# if_exists="replace" -> overwrite if table already exists
# index=False        -> do not save the row numbers
students.to_sql("students", db, if_exists="replace", index=False)

print("Table 'students' saved in the database!")

Table 'students' saved in the database!


In [26]:
# Read it back from the database
pd.read_sql("SELECT * FROM students", db)

,roll_no,name,marks
0,101,Aman,78.5
1,102,Bhavna,85.0
2,103,Chirag,62.0
3,104,Divya,91.5


In [27]:
# SQLite keeps a list of all tables in a special table called sqlite_master
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", db)

,name
0,person
1,students


In [28]:
con.close()        # close the sqlite3 connection
db.dispose()       # close the SQLAlchemy engine

if os.path.exists("mydata.sqlite"):
    os.remove("mydata.sqlite")
    print("Deleted: mydata.sqlite")

Deleted: mydata.sqlite
